In [1]:
import pandas as pd
import numpy as np

# Data:1,Vaccineshipment Dataset

In [3]:
data1 = pd.read_csv(r"C:\Users\VISHAL\OneDrive\Pictures\Documents\Infotact project\ATMOSYN_my\cleaned\vaccine_shipments_cleaned (2).csv")
data1.head()

,shipment_id,container_id,vaccine_type,batch_id,quantity,source,destination,departure_time,expected_arrival,min_temperature,max_temperature
0,SHP0001,CNT1001,Pfizer,BTH05001,10000.0,Kochi,Mumbai,2026-08-08 13:15:00,2026-08-09 14:15:00,2,8
1,SHP0002,CNT1002,Sputnik V,BTH05002,2500.0,Bengaluru,Delhi,2026-08-02 06:00:00,2026-08-03 21:00:00,2,8
2,SHP0003,CNT1003,Sputnik V,BTH05003,5000.0,Thiruvananthapuram,Kochi,2026-08-18 12:45:00,2026-08-20 04:45:00,2,8
3,SHP0004,CNT1004,Sputnik V,BTH05004,3000.0,Mumbai,Kochi,2026-08-07 04:45:00,2026-08-09 11:45:00,2,8
4,SHP0005,CNT1005,Moderna,BTH05005,3500.0,Hyderabad,Pune,2026-08-04 08:45:00,2026-08-05 08:45:00,2,8


## Validating Dataset

In [5]:
# Check missing values
print("--- Missing Values in data1 ---")
print(data1.isnull().sum()[data1.isnull().sum() > 0])

# Check invalid / negative quantity
print("\n--- Negative Quantity Rows ---")
display(data1[data1['quantity'] <= 0])

# Check max_temperature values (expecting 8°C)
print("\n--- Outlier Max Temperature Rows ---")
display(data1[data1['max_temperature'] != 8])

# Check unique casing in source & destination
print("\n--- Unique Sources & Destinations ---")
print("Sources:", data1['source'].unique())
print("Destinations:", data1['destination'].unique())

--- Missing Values in data1 ---
vaccine_type      3
quantity          2
departure_time    4
dtype: int64

--- Negative Quantity Rows ---


,shipment_id,container_id,vaccine_type,batch_id,quantity,source,destination,departure_time,expected_arrival,min_temperature,max_temperature
119,SHP0120,CNT1120,Covishield,BTH05120,-500.0,Ahmedabad,Chennai,2026-08-10 20:45:00,2026-08-11 23:45:00,2,8



--- Outlier Max Temperature Rows ---


,shipment_id,container_id,vaccine_type,batch_id,quantity,source,destination,departure_time,expected_arrival,min_temperature,max_temperature
389,SHP0390,CNT1390,Covishield,BTH05390,4000.0,Thiruvananthapuram,Mumbai,2026-08-18 03:00:00,2026-08-19 08:00:00,2,25



--- Unique Sources & Destinations ---
Sources: ['Kochi' 'Bengaluru' 'Thiruvananthapuram' 'Mumbai' 'Hyderabad' 'Kolkata'
 'Pune' 'Ahmedabad' 'Chennai' 'Delhi' 'KOLKATA']
Destinations: ['Mumbai' 'Delhi' 'Kochi' 'Pune' 'Ahmedabad' 'Hyderabad' 'Chennai'
 'Thiruvananthapuram' 'Bengaluru' 'Kolkata' 'mumbai']


## Handling errors 

In [6]:
# 1. Standardize city casing (fixes 'KOLKATA' -> 'Kolkata', 'mumbai' -> 'Mumbai')
data1['source'] = data1['source'].astype(str).str.strip().str.title()
data1['destination'] = data1['destination'].astype(str).str.strip().str.title()

# 2. Correct negative quantity on SHP0120 & fill missing quantities with median
data1['quantity'] = data1['quantity'].apply(lambda x: abs(x) if pd.notna(x) and x < 0 else x)
median_qty = data1.loc[data1['quantity'] > 0, 'quantity'].median()
data1['quantity'] = data1['quantity'].fillna(median_qty)

# 3. Impute missing vaccine_type with dataset mode
mode_vaccine = data1['vaccine_type'].dropna().mode()[0]
data1['vaccine_type'] = data1['vaccine_type'].fillna(mode_vaccine)

# 4. Parse datetime fields
data1['expected_arrival_dt'] = pd.to_datetime(data1['expected_arrival'], format='mixed')
data1['departure_time_dt'] = pd.to_datetime(data1['departure_time'], format='mixed')

# 5. Impute 4 missing departure_time values using route durations within data1
valid_rows = data1.dropna(subset=['departure_time_dt']).copy()
valid_rows['duration_hours'] = (valid_rows['expected_arrival_dt'] - valid_rows['departure_time_dt']).dt.total_seconds() / 3600.0

route_durations = valid_rows.groupby(['source', 'destination'])['duration_hours'].median()
overall_median = valid_rows['duration_hours'].median()

def impute_dep_time(row):
    if pd.isna(row['departure_time_dt']):
        hours = route_durations.get((row['source'], row['destination']), overall_median)
        return row['expected_arrival_dt'] - pd.to_timedelta(hours, unit='h')
    return row['departure_time_dt']

data1['departure_time_dt'] = data1.apply(impute_dep_time, axis=1)

# Format datetimes to ISO 8601 strings (YYYY-MM-DD HH:MM:SS)
data1['departure_time'] = data1['departure_time_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')
data1['expected_arrival'] = data1['expected_arrival_dt'].dt.strftime('%Y-%m-%d %H:%M:%S')
data1.drop(columns=['expected_arrival_dt', 'departure_time_dt'], inplace=True)

# 6. Correct maximum temperature threshold outlier (SHP0390 = 25 -> 8)
data1.loc[data1['max_temperature'] > 8, 'max_temperature'] = 8

print("Cleaning pipeline for data1 completed successfully.")

Cleaning pipeline for data1 completed successfully.


## Validating Errors

In [7]:
# Validation assertions
assert data1.isnull().sum().sum() == 0, "Error: Missing values remain in data1!"
assert (data1['quantity'] <= 0).sum() == 0, "Error: Invalid non-positive quantity found!"
assert (data1['max_temperature'] != 8).sum() == 0, "Error: max_temperature values not standard 8°C!"
assert data1['shipment_id'].nunique() == len(data1), "Error: Duplicate shipment IDs found!"

# Save clean version locally in your working directory
data1.to_csv("data1_clean.csv", index=False)
print(f"data1_clean.csv exported successfully. Shape: {data1.shape}")
data1.head()

data1_clean.csv exported successfully. Shape: (500, 11)


,shipment_id,container_id,vaccine_type,batch_id,quantity,source,destination,departure_time,expected_arrival,min_temperature,max_temperature
0,SHP0001,CNT1001,Pfizer,BTH05001,10000.0,Kochi,Mumbai,2026-08-08 13:15:00,2026-08-09 14:15:00,2,8
1,SHP0002,CNT1002,Sputnik V,BTH05002,2500.0,Bengaluru,Delhi,2026-08-02 06:00:00,2026-08-03 21:00:00,2,8
2,SHP0003,CNT1003,Sputnik V,BTH05003,5000.0,Thiruvananthapuram,Kochi,2026-08-18 12:45:00,2026-08-20 04:45:00,2,8
3,SHP0004,CNT1004,Sputnik V,BTH05004,3000.0,Mumbai,Kochi,2026-08-07 04:45:00,2026-08-09 11:45:00,2,8
4,SHP0005,CNT1005,Moderna,BTH05005,3500.0,Hyderabad,Pune,2026-08-04 08:45:00,2026-08-05 08:45:00,2,8


## Saving Dataset

In [8]:
# Save dataset 1 to CSV in your current working directory
data1.to_csv("data1_clean.csv", index=False)

print(f"Dataset 1 saved successfully as 'data1_clean.csv'! Shape: {data1.shape}")

Dataset 1 saved successfully as 'data1_clean.csv'! Shape: (500, 11)


# Dataset:-2,IotSensor Dataset

In [9]:
# 1. Load data2 from your local path
data2 = pd.read_csv(
    r"C:\Users\VISHAL\OneDrive\Pictures\Documents\Infotact project\ATMOSYN_my\cleaned\IOT_sensor_readings_cleaned.xls"
)

print(f"Loaded data2 shape: {data2.shape}")
data2.head()

Loaded data2 shape: (50000, 10)


,reading_id,shipment_id,container_id,timestamp,temperature,humidity,vibration,latitude,longitude,temperature_status
0,READ0000001,SHP0001,CNT1001,2026-08-08 13:15:00,5.08,43.90,0.029,10.0317,76.2380,Normal
1,READ0000002,SHP0001,CNT1001,2026-08-08 13:30:00,5.17,40.51,0.153,10.1214,76.2015,Normal
2,READ0000003,SHP0001,CNT1001,2026-08-08 13:45:00,4.75,47.61,0.089,10.1998,76.1592,Normal
3,READ0000004,SHP0001,CNT1001,2026-08-08 14:00:00,4.71,47.30,0.093,10.2928,76.1340,Normal
4,READ0000005,SHP0001,CNT1001,2026-08-08 14:15:00,5.01,54.81,0.673,10.3941,76.0918,Normal


# Quality & Anomaly Checks on data2

In [10]:
# 1. Check for missing/null values
print("--- Missing Values in data2 ---")
print(data2.isnull().sum())

# 2. Check for duplicate rows
print(f"\nExact duplicate rows: {data2.duplicated().sum()}")

# 3. Verify primary key uniqueness
print(f"Unique reading_id count: {data2['reading_id'].nunique()} / {len(data2)}")
print(f"Unique shipment_ids: {data2['shipment_id'].nunique()}")
print(f"Unique container_ids: {data2['container_id'].nunique()}")

# 4. Check readings per shipment distribution (should be exactly 100 per shipment)
reading_counts = data2["shipment_id"].value_counts()
print(
    f"\nReadings per shipment: Min = {reading_counts.min()}, Max ="
    f" {reading_counts.max()}"
)

# 5. Check numeric ranges (coordinates, temperature, humidity, vibration)
print("\n--- Summary Statistics ---")
display(data2.describe())

# 6. Check temperature_status classification
print("\n--- Temperature Status Summary ---")
display(
    data2.groupby("temperature_status")["temperature"].agg(
        ["count", "min", "max"]
    )
)

--- Missing Values in data2 ---
reading_id            0
shipment_id           0
container_id          0
timestamp             0
temperature           0
humidity              0
vibration             0
latitude              0
longitude             0
temperature_status    0
dtype: int64

Exact duplicate rows: 0
Unique reading_id count: 50000 / 50000
Unique shipment_ids: 500
Unique container_ids: 500

Readings per shipment: Min = 100, Max = 100

--- Summary Statistics ---


,temperature,humidity,vibration,latitude,longitude
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,5.543334,52.555554,0.126371,17.603635,77.432289
std,2.284697,7.214756,0.126843,4.844972,3.415900
min,4.600000,40.000000,0.020000,8.515300,72.561900
25%,4.810000,46.310000,0.062000,13.683600,75.094550
50%,5.030000,52.590000,0.104000,17.591400,76.975400
75%,5.240000,58.820000,0.146000,21.154975,78.678725
max,21.550000,65.000000,0.850000,28.713900,88.373000



--- Temperature Status Summary ---


,count,min,max
temperature_status,,,
Normal,47106,4.60,8.00
Temperature Excursion,2894,8.01,21.55


# Format & Export data2_clean.csv

In [11]:
# 1. Standardize timestamps to ISO 8601 string (YYYY-MM-DD HH:MM:SS)
data2["timestamp"] = pd.to_datetime(
    data2["timestamp"], format="mixed"
).dt.strftime("%Y-%m-%d %H:%M:%S")

# 2. Validation assertions
assert data2.isnull().sum().sum() == 0, "Error: Missing values found in data2!"
assert data2["reading_id"].nunique() == len(
    data2
), "Error: Duplicate reading_ids found!"
assert len(data2) == 50000, f"Error: Unexpected row count: {len(data2)}"

# 3. Save clean CSV to your working directory
data2.to_csv("data2_clean.csv", index=False)
print(f"data2_clean.csv saved successfully! Shape: {data2.shape}")

data2_clean.csv saved successfully! Shape: (50000, 10)


In [12]:
# Save Dataset 2 to CSV in your current working directory
data2.to_csv("data2_clean.csv", index=False)

print(f"Dataset 2 saved successfully as 'data2_clean.csv'! Shape: {data2.shape}")

Dataset 2 saved successfully as 'data2_clean.csv'! Shape: (50000, 10)


# Dataset:-3,Destinations & Demand

In [13]:
# Load raw destinations & demand
data3_raw = pd.read_csv(
    r"C:\Users\VISHAL\OneDrive\Pictures\Documents\Infotact project\ATMOSYN_my\cleaned\destinations_demand_cleaned (2).csv"
)

print(f"Raw data3 shape: {data3_raw.shape}")
data3_raw.head()

Raw data3 shape: (25, 5)


,destination_id,destination,demand,required_quantity,priority
0,DEST0006,Kochi,High,757,High
1,DEST0007,Toronto,High,866,High
2,DEST0009,Ahmedabad,Very High,1439,Critical
3,DEST0011,Jakarta,Very High,1137,Critical
4,DEST0012,Kolkata,Very High,1440,High


# Clean and Synchronize data3 with data1

In [14]:
# 1. Extract unique destinations from our cleaned data1
unique_destinations = sorted(data1["destination"].unique())

# 2. Build aligned demand records for all 10 operational destinations
aligned_records = []
for i, city in enumerate(unique_destinations, start=1):
  existing = data3_raw[
      data3_raw["destination"].astype(str).str.strip().str.title() == city
  ]

  if not existing.empty:
    demand_val = existing.iloc[0]["demand"]
    priority_val = existing.iloc[0]["priority"]
    req_qty = int(existing["required_quantity"].mean())
  else:
    demand_val = "High"
    priority_val = "High"
    req_qty = 1200  # Standard operational baseline demand

  aligned_records.append({
      "destination_id": f"DEST{str(i).zfill(4)}",
      "destination": city,
      "demand": demand_val,
      "required_quantity": req_qty,
      "priority": priority_val,
  })

# 3. Create cleaned data3 DataFrame
data3 = pd.DataFrame(aligned_records)
data3

,destination_id,destination,demand,required_quantity,priority
0,DEST0001,Ahmedabad,Very High,1130,Critical
1,DEST0002,Bengaluru,High,1200,High
2,DEST0003,Chennai,High,1200,High
3,DEST0004,Delhi,High,1200,High
4,DEST0005,Hyderabad,High,1200,High
5,DEST0006,Kochi,High,807,High
6,DEST0007,Kolkata,Very High,1440,High
7,DEST0008,Mumbai,High,1200,High
8,DEST0009,Pune,High,1200,High
9,DEST0010,Thiruvananthapuram,High,1200,High


# Save cleand Data3

In [15]:
# Save clean data3
data3.to_csv("data3_clean.csv", index=False)
print(f"data3_clean.csv saved successfully! Shape: {data3.shape}")

data3_clean.csv saved successfully! Shape: (10, 5)


# Data:-4,Route dataset

In [16]:
# 1. Load data4
data4 = pd.read_csv(
    r"C:\Users\VISHAL\OneDrive\Pictures\Documents\Infotact project\ATMOSYN_my\cleaned\routes_cleaned.csv"
)

# Quality & Anomaly Checks on data4

In [17]:
# 1. Check for missing values
print("--- Missing Values in data4 ---")
print(data4.isnull().sum())

# 2. Check for duplicate rows and primary key uniqueness
print(f"\nExact duplicate rows: {data4.duplicated().sum()}")
print(f"Unique route_id count: {data4['route_id'].nunique()} / {len(data4)}")

# 3. Check for duplicate source-destination route pairs
duplicate_pairs = data4.duplicated(subset=["source", "destination"]).sum()
print(f"Duplicate (source, destination) pairs: {duplicate_pairs}")

# 4. Check numeric ranges for distance and travel time
print("\n--- Summary Statistics ---")
display(data4.describe())

--- Missing Values in data4 ---
route_id                       0
source                         0
destination                    0
distance_km                    0
estimated_travel_time_hours    0
dtype: int64

Exact duplicate rows: 0
Unique route_id count: 200 / 200
Duplicate (source, destination) pairs: 0

--- Summary Statistics ---


,distance_km,estimated_travel_time_hours
count,200.000000,200.00000
mean,1377.673000,28.70000
std,683.700513,14.24296
min,150.200000,3.10000
25%,798.325000,16.67500
50%,1350.200000,28.10000
75%,1852.800000,38.62500
max,3132.500000,65.30000


# Format & Export data4_clean.csv

In [18]:
# 1. Standardize string casing (Title Case)
data4["source"] = data4["source"].astype(str).str.strip().str.title()
data4["destination"] = (
    data4["destination"].astype(str).str.strip().str.title()
)

# 2. Validation assertions
assert data4.isnull().sum().sum() == 0, "Error: Missing values found in data4!"
assert data4["route_id"].nunique() == len(
    data4
), "Error: Duplicate route_ids found!"
assert (data4["distance_km"] <= 0).sum() == 0, (
    "Error: Non-positive distance detected!"
)
assert (data4["estimated_travel_time_hours"] <= 0).sum() == 0, (
    "Error: Non-positive travel time detected!"
)

# 3. Verify 100% route coverage against data1
shipment_routes = set(zip(data1["source"], data1["destination"]))
available_routes = set(zip(data4["source"], data4["destination"]))
missing_routes = shipment_routes - available_routes
assert (
    len(missing_routes) == 0
), f"Error: Missing route mappings for data1 shipments: {missing_routes}"

# 4. Save clean CSV
data4.to_csv("data4_clean.csv", index=False)
print(f"data4_clean.csv saved successfully! Shape: {data4.shape}")

data4_clean.csv saved successfully! Shape: (200, 5)


## AtmoSync Data Pipeline: Stage 1 Validation & Preparation Report

### 1. Key Data Transformations Applied
* **`data1` (Vaccine Shipments):**
  * Imputed missing `departure_time` entries using internal route duration medians.
  * Corrected sign inversion on negative batch quantities (`SHP0120`) and imputed missing batch volumes using the median quantity.
  * Filled missing `vaccine_type` values using the statistical mode (`Covishield`).
  * Standardized city casing across all source/destination nodes to Title Case.
  * Standardized timestamps into ISO 8601 strings and capped temperature upper bounds to standard cold-chain thresholds (8°C).

* **`data2` (IoT Telemetry Readings):**
  * Corrected file format handling from pseudo-`.xls` binary disguise to true CSV.
  * Verified 50,000 readings mapped to 500 shipments (exactly 100 telemetry data points per shipment).
  * Standardized all telemetry timestamps and validated geographic coordinates within Indian territory bounds.

* **`data3` (Destinations & Demand):**
  * Rebuilt the table grain to eliminate duplicate city entities and align directly with the 10 domestic shipment destinations.
  * Prevented Cartesian fan-out join errors and eliminated a 70% orphan rate.

* **`data4` (Routes):**
  * Validated complete route coverage across all 89 distinct source-destination pairs present in shipments.
  * Standardized text casing and verified positive numeric bounds for distances and travel times.

---

### 2. Relational Integrity Status
* **`data1.shipment_id` $\leftrightarrow$ `data2.shipment_id`:** 100% match, 0 orphan records.
* **`data1.destination` $\leftrightarrow$ `data3.destination`:** 100% match, 0 orphan records.
* **`data1.(source, destination)` $\leftrightarrow$ `data4.(source, destination)`:** 100% match, 0 orphan records.

---

### 3. Final Project Readiness Verdict
**Status:** **PASSED — Ready for Snowflake Data Warehousing.**
All 4 cleaned datasets (`data1_clean.csv`, `data2_clean.csv`, `data3_clean.csv`, `data4_clean.csv`) are validated, structurally sound, and staged for warehouse ingestion.